# Factor Model: Step 4a - Alpha 101 Calculation

## Objective
Calculate Alpha 101 factors and save results to avoid re-calculation.

### Workflow:
1. Load winsorized data from Step 3
2. Convert to panel format
3. Calculate VWAP
4. Load Alpha 101 functions
5. Calculate all 101 alpha factors
6. **Save Alpha 101 results to disk**

### Outputs:
- `alpha101_results.parquet` - Successfully calculated factors
- `alpha101_errors.csv` - Errors log
- `data_panels.parquet` - Data dictionary for Step 4b

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('.')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Load Winsorized Data from Step 3

In [ ]:
data_path = 'russell2000_winsorized_step3.parquet'

print("Loading winsorized data...")
df = pd.read_parquet(data_path)

print(f"✓ Data loaded successfully")
print(f"  Shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  Date range: {df.index.get_level_values(0).min()} to {df.index.get_level_values(0).max()}")
print(f"  Unique symbols: {df.index.get_level_values(1).nunique():,}")

# Optimize memory by converting to float32
print("\n  Optimizing data types to reduce memory...")
for col in ['open', 'high', 'low', 'close', 'volume', 'avg_price', 'return_winsorized']:
    if col in df.columns:
        df[col] = df[col].astype('float32')

print(f"  Memory after optimization: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nFirst few rows:")
display(df.head(10))

## 2. Convert to Panel Format

In [ ]:
print("Converting to panel format (date × symbol)...")
print("  ⚠ This may take a few minutes for large datasets...\n")

# Extract each field as a panel (date × symbol)
open_panel = df['open'].unstack(level=1)
high_panel = df['high'].unstack(level=1)
low_panel = df['low'].unstack(level=1)
close_panel = df['close'].unstack(level=1)
volume_panel = df['volume'].unstack(level=1)
returns_panel = df['return_winsorized'].unstack(level=1)

# Use the actual avg_price from Step 2 (20-day rolling average of close)
# This is better than approximating VWAP from daily OHLC data
vwap_panel = df['avg_price'].unstack(level=1)

# Clean up original dataframe to free memory
del df
import gc
gc.collect()

print(f"✓ Panel format created")
print(f"  Panel shape: {close_panel.shape}")
print(f"  Dates: {len(close_panel)}")
print(f"  Symbols: {len(close_panel.columns)}")
print(f"  Memory per panel: ~{close_panel.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 3. Use Existing Average Price (VWAP Proxy)

**Note**: Your data already contains `avg_price` from Step 2, which is a 20-day rolling average of the closing price. This is more appropriate than approximating VWAP from daily OHLC bars, since true VWAP requires intraday tick data.

In [ ]:
print("Using existing avg_price as VWAP proxy...")
print("Source: 20-day rolling average of close price from Step 2\n")

print(f"✓ VWAP panel extracted")
print(f"  Shape: {vwap_panel.shape}")
print(f"\nVWAP Statistics:")
print(vwap_panel.stack().describe())

## 4. Create Data Dictionary

In [ ]:
print("Preparing data dictionary...\n")

data = {
    'open': open_panel,
    'high': high_panel,
    'low': low_panel,
    'close': close_panel,
    'volume': volume_panel,
    'vwap': vwap_panel,
    'returns': returns_panel,
    'avg_price': vwap_panel
}

print(f"✓ Data dictionary created")
print(f"  Keys: {list(data.keys())}")
print(f"  All panels shape: {data['close'].shape}")

## 5. Save Data Panels for Step 4b

Save the data panels so Step 4b doesn't need to recalculate them.

In [ ]:
print("Saving data panels for Step 4b...")
print("  Converting to stacked format...")

# Save all panels to a single parquet file with multi-level columns
# Use float32 to save space
panels_combined = pd.concat([
    open_panel.stack().rename('open'),
    high_panel.stack().rename('high'),
    low_panel.stack().rename('low'),
    close_panel.stack().rename('close'),
    volume_panel.stack().rename('volume'),
    vwap_panel.stack().rename('vwap'),
    returns_panel.stack().rename('returns')
], axis=1)

# Ensure all columns are float32
for col in panels_combined.columns:
    panels_combined[col] = panels_combined[col].astype('float32')

print("  Writing to disk...")
panels_combined.to_parquet('data_panels.parquet', compression='snappy')

# Clean up to free memory
del panels_combined
gc.collect()

import os
file_size_mb = os.path.getsize('data_panels.parquet') / 1024**2
print(f"✓ Data panels saved to data_panels.parquet")
print(f"  Size: {file_size_mb:.2f} MB")

## 6. Load Alpha 101 Functions

In [ ]:
print("Loading Alpha 101 functions...\n")

import json
import nbformat
from IPython import get_ipython

with open('alpha101_factors.ipynb', 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

ipython = get_ipython()
cell_count = 0

for i, cell in enumerate(nb.cells):
    if cell.cell_type == 'code':
        code = ''.join(cell.source) if isinstance(cell.source, list) else cell.source
        
        if not code.strip() or 'Example usage:' in code or 'df_open' in code:
            continue
            
        try:
            ipython.run_cell(code, store_history=False, silent=True)
            cell_count += 1
        except Exception as e:
            print(f"  Warning: Cell {i} failed: {str(e)[:60]}")

print(f"✓ Alpha 101 notebook loaded ({cell_count} cells executed)")

alpha101_functions = [f'alpha{i:03d}' for i in range(1, 102)]
available_alpha101 = [f for f in alpha101_functions if f in dir()]

print(f"  Available Alpha 101 functions: {len(available_alpha101)}/101")
print(f"  Sample: {available_alpha101[:5]}")

## 7. Calculate Alpha 101 Factors

In [ ]:
print("="*80)
print("CALCULATING ALPHA 101 FACTORS")
print("="*80)
print("\nThis will take several minutes...\n")

alpha101_results = {}
alpha101_errors = {}

for i in tqdm(range(1, 102), desc="Alpha 101"):
    func_name = f'alpha{i:03d}'
    
    try:
        if func_name in dir():
            alpha_func = eval(func_name)
            result = alpha_func(data)
            
            # Handle 3D array results
            if isinstance(result, np.ndarray) and result.ndim == 3:
                result = pd.DataFrame(result[0], index=data['close'].index, columns=data['close'].columns)
            
            alpha101_results[f'alpha101_{i:03d}'] = result
        else:
            alpha101_errors[f'alpha101_{i:03d}'] = "Function not found"
            
    except Exception as e:
        alpha101_errors[f'alpha101_{i:03d}'] = str(e)

print(f"\n✓ Alpha 101 calculation complete")
print(f"  Successful: {len(alpha101_results)}/101")
print(f"  Errors: {len(alpha101_errors)}")

if alpha101_errors:
    print(f"\n⚠ Errors encountered:")
    for name, error in list(alpha101_errors.items())[:10]:
        print(f"  {name}: {error[:60]}...")

## 8. Save Alpha 101 Results

In [ ]:
print("="*80)
print("SAVING ALPHA 101 RESULTS")
print("="*80)

if len(alpha101_results) > 0:
    print(f"\nSaving {len(alpha101_results)} Alpha 101 factors...")
    
    # Convert to DataFrame format for saving
    # Stack all results into a single DataFrame
    alpha101_dfs = []
    
    print("  Processing results...")
    for factor_name, factor_values in alpha101_results.items():
        if isinstance(factor_values, pd.Series):
            df_temp = factor_values.to_frame(name=factor_name)
        elif isinstance(factor_values, pd.DataFrame):
            if len(factor_values.columns) > 1:
                df_temp = factor_values.stack().to_frame(name=factor_name)
            else:
                df_temp = factor_values.rename(columns={factor_values.columns[0]: factor_name})
        
        # Convert to float32 to save memory
        df_temp = df_temp.astype('float32')
        alpha101_dfs.append(df_temp)
    
    print("  Concatenating results...")
    df_alpha101 = pd.concat(alpha101_dfs, axis=1)
    
    # Save to parquet
    print("  Writing to disk...")
    df_alpha101.to_parquet('alpha101_results.parquet', compression='snappy')
    
    # Clean up
    del df_alpha101, alpha101_dfs
    gc.collect()
    
    file_size_mb = os.path.getsize('alpha101_results.parquet') / 1024**2
    print(f"✓ Alpha 101 results saved to alpha101_results.parquet")
    print(f"  Size: {file_size_mb:.2f} MB")
else:
    print("❌ No Alpha 101 results to save!")

# Save errors log
if alpha101_errors:
    error_df = pd.DataFrame([
        {'factor': k, 'error': v} for k, v in alpha101_errors.items()
    ])
    error_df.to_csv('alpha101_errors.csv', index=False)
    print(f"\n✓ Error log saved to alpha101_errors.csv")
    print(f"  {len(alpha101_errors)} errors logged")

## Summary

In [ ]:
print("="*80)
print("STEP 4A COMPLETE - ALPHA 101 CALCULATED")
print("="*80)

print(f"\nResults:")
print(f"  Successfully calculated: {len(alpha101_results)}/101")
print(f"  Errors: {len(alpha101_errors)}")
print(f"  Success rate: {len(alpha101_results)/101*100:.1f}%")

print(f"\nOutput files:")
print(f"  ✓ alpha101_results.parquet - Alpha 101 factor loadings")
print(f"  ✓ data_panels.parquet - Data for Step 4b")
if alpha101_errors:
    print(f"  ✓ alpha101_errors.csv - Error log")

print(f"\n" + "="*80)
print("Next: Run Step 4b to calculate Alpha 179 factors")
print("="*80)